In [0]:
%sql
CREATE OR REPLACE TABLE silver.sales
USING DELTA
AS
SELECT
  order_number,
  product_key,
  customer_key,
 
  -- order_date has 19 NULL rows
  -- We fill them using shipping_date minus 7 days as an estimate
  COALESCE(
    CAST(order_date AS DATE),
    DATE_SUB(CAST(shipping_date AS DATE), 7)
  )                                      AS order_date,
 
  CAST(shipping_date AS DATE)          AS shipping_date,
  CAST(due_date AS DATE)              AS due_date,
 
  -- How many days from order to shipping?
  DATEDIFF(
    CAST(shipping_date AS DATE),
    CAST(order_date AS DATE)
  )                                      AS days_to_ship,
 
  sales_amount,
  quantity,
  price,
 
  -- Extract year and month for easy reporting later
  YEAR(CAST(order_date AS DATE))     AS order_year,
  MONTH(CAST(order_date AS DATE))    AS order_month
 
FROM workspace.bronze.sales;


In [0]:
%sql
-- Verify: confirm order_date has 0 NULLs now
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN order_date IS NULL THEN 1 ELSE 0 END) AS null_dates
FROM workspace.silver.sales;
